# Business Failures Corpus Exploration

This notebook studies portable corpus exports rather than the live SQLite or Chroma internals. Run `topic_corpus.py export` first. In Google Colab, upload the exported data folder or mount Google Drive and change `EXPORT_DIR`.

In [1]:
from pathlib import Path
import os
import pandas as pd

if os.environ.get("TOPIC_EXPORT_DIR"):
    EXPORT_DIR = Path(os.environ["TOPIC_EXPORT_DIR"])
else:
    export_candidates = [
        Path(".workspace/topics/business-failures/exports"),
        Path("../.workspace/topics/business-failures/exports"),
    ]
    EXPORT_DIR = next((path for path in export_candidates if path.exists()), export_candidates[0])
if not EXPORT_DIR.exists():
    raise FileNotFoundError(
        f"Export folder not found: {EXPORT_DIR.resolve()}\n"
        "Run the export command locally or upload the data in Colab."
    )
passages = pd.read_parquet(EXPORT_DIR / "passages.parquet")
cases = pd.read_csv(EXPORT_DIR / "cases.csv")
case_mechanisms = pd.read_csv(EXPORT_DIR / "case_mechanisms.csv")
print(f"{len(cases)} cases; {len(passages)} passages")

20 cases; 388 passages


## 1. Corpus inventory and completeness

In [2]:
display(cases[["subject", "subject_type", "industry", "time_period"]])
display(passages.groupby(["review_status", "is_sponsor"]).size().rename("passages"))

,subject,subject_type,industry,time_period
0,Pizza Hut,company,restaurants,1958-2026
1,American Airlines,company,airlines,1930-2026
2,NASA,government_agency,space,1958-2026
3,Tesla,company,automotive,2010-2026
4,FTX,company,cryptocurrency,2017-2024
5,McDonald's,company,restaurants,1940-2026
6,Snapchat,company,social_media,2011-2026
7,Doritos,product_brand,packaged_food,1964-2026
8,Craigslist,company,online_classifieds,1995-2026
9,Pickleball business ecosystem,market_ecosystem,sports_and_recreation,1965-2026


review_status   is_sponsor
curated_workup  False          22
machine_workup  False         312
                True            1
rule_applied    True           53
Name: passages, dtype: int64

## 2. Failure mechanisms by case

In [3]:
matrix = pd.crosstab(
    case_mechanisms["subject"],
    case_mechanisms["failure_mechanism"],
)
display(matrix)

failure_mechanism,acquisition_or_integration_failure,adverse_selection,capability_erosion,channel_or_platform_dependency,commoditization,cost_structure_mismatch,failed_transformation,feedback_suppression,financial_leverage,fraud_or_asset_misappropriation,...,normalization_of_deviance,overexpansion_or_overcapacity,ownership_or_portfolio_neglect,path_dependence,quality_or_value_erosion,short_term_optimization,stakeholder_betrayal,strategic_drift,success_induced_obsolescence,unclear_positioning
subject,,,,,,,,,,,,,,,,,,,,,
American Airlines,0,0,1,0,0,0,1,0,1,0,...,0,0,0,0,0,1,1,0,0,1
American Idol,0,0,0,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
Applebee's,0,0,0,0,0,1,1,0,1,0,...,0,0,1,1,0,0,0,0,0,1
Cartoon Network,0,0,1,1,0,0,1,0,0,0,...,0,0,1,0,0,1,1,0,0,0
Craigslist,0,0,1,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
Crocs,1,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
Doritos,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,1,1,0,0
FTX,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,1,0,0,0
Five Guys,0,0,0,0,0,1,0,0,0,0,...,0,1,0,1,1,0,0,0,0,1


## 3. Passage-label frequency and co-occurrence

In [4]:
content = passages[passages["include_in_index"]].copy()
mechanisms = content.assign(
    failure_mechanism=content["failure_mechanisms"].fillna("").str.split("|")
).explode("failure_mechanism")
mechanisms = mechanisms[mechanisms["failure_mechanism"] != ""]
display(mechanisms["failure_mechanism"].value_counts().to_frame("passages"))

,passages
failure_mechanism,
market_or_demographic_shift,53
mission_loss,42
incentive_misalignment,42
governance_failure,39
cost_structure_mismatch,32
quality_or_value_erosion,30
strategic_drift,29
ownership_or_portfolio_neglect,26
identity_or_reputation_coupling,19


## 4. Stratified review sample

Use this sample to check machine labels, transcript quality, and whether the summary is faithful to the source passage.

In [5]:
review_sample = (
    content.groupby("video_id", group_keys=False)
    .apply(lambda frame: frame.sample(min(2, len(frame)), random_state=23), include_groups=False)
)
display(review_sample[[
    "subject", "summary", "failure_mechanisms", "evidence_types", "deep_link"
]])

,subject,summary,failure_mechanisms,evidence_types,deep_link
181,Craigslist,"Craigslist's founder, Craig Newmark, initially...",short_term_optimization|financial_leverage,narrator_analysis,https://www.youtube.com/watch?v=31oqfd8myBI&t=0s
193,Craigslist,Philip Knowlton's sale of shares to eBay led t...,ownership_or_portfolio_neglect,historical_event,https://www.youtube.com/watch?v=31oqfd8myBI&t=...
93,FTX,The passage recounts Sam Bankman-Fried's claim...,fraud_or_asset_misappropriation,historical_event,https://www.youtube.com/watch?v=3470164O6p8&t=0s
109,FTX,"The actions of FTX's competitor, Binance, led ...",incentive_misalignment|stakeholder_betrayal,narrator_analysis|named_source_or_quote,https://www.youtube.com/watch?v=3470164O6p8&t=...
303,Shark Tank,Kevin O'Leary's involvement in the FTX scandal...,identity_or_reputation_coupling|governance_fai...,historical_event,https://www.youtube.com/watch?v=3cQ20dNnmDA&t=...
294,Shark Tank,The success of the show Shark Tank significant...,,historical_event,https://www.youtube.com/watch?v=3cQ20dNnmDA&t=...
323,Five Guys,Five Guys Burgers once reached $1.3 billion in...,,quantitative_metric|named_source_or_quote,https://www.youtube.com/watch?v=4-8_etpRj1Y&t=0s
324,Five Guys,"Jerry Morrell, a financial planner, decided to...",,narrator_analysis,https://www.youtube.com/watch?v=4-8_etpRj1Y&t=64s
0,Pizza Hut,"Pizza Hut, once the largest pizza provider, ha...",strategic_drift,historical_event,https://www.youtube.com/watch?v=CU3xYnICJ0s&t=0s
13,Pizza Hut,,unclear_positioning,,https://www.youtube.com/watch?v=CU3xYnICJ0s&t=...


## 5. Candidate teaching themes

A frequent label is not automatically a lesson. Review the passages and add: the observation, boundary conditions, a counterexample, and what additional evidence would change the conclusion.

In [6]:
theme = "short_term_optimization"
theme_passages = mechanisms[mechanisms["failure_mechanism"] == theme]
display(theme_passages[["subject", "summary", "text", "deep_link"]].head(20))

,subject,summary,text,deep_link
4,Pizza Hut,,While Domino's continued to be run by people w...,https://www.youtube.com/watch?v=CU3xYnICJ0s&t=...
19,Pizza Hut,,It's for real. There are some fundamental prob...,https://www.youtube.com/watch?v=CU3xYnICJ0s&t=...
28,American Airlines,,to artificially keep prices high. It's called ...,https://www.youtube.com/watch?v=KhoAf9K9JC8&t=...
35,American Airlines,While competitors improved their business mode...,"employment agreements, lease agreements on gat...",https://www.youtube.com/watch?v=KhoAf9K9JC8&t=...
40,American Airlines,American Airlines spent nearly $13 billion on ...,which is basically handing money to shareholde...,https://www.youtube.com/watch?v=KhoAf9K9JC8&t=...
44,American Airlines,The author reflects on negative experiences fl...,"of the big four carriers, if you include South...",https://www.youtube.com/watch?v=KhoAf9K9JC8&t=...
45,American Airlines,The author asserts that past management decisi...,"including me, it's because of all those decisi...",https://www.youtube.com/watch?v=KhoAf9K9JC8&t=...
135,McDonald's,McDonald's discounted their offerings to recov...,there's no predators and no humans methyl. So ...,https://www.youtube.com/watch?v=v8w6djdUt1U&t=...
174,Doritos,Frito-Lay's management made decisions that pri...,became a rallying crawl with Gen Z going on Ti...,https://www.youtube.com/watch?v=WhVSww5fzBA&t=...
179,Doritos,Elliott Management pressured PepsiCo to improv...,seeing the mess that was going on at PepsiCo. ...,https://www.youtube.com/watch?v=WhVSww5fzBA&t=...


In [ ]:
# Synthesize candidate teaching themes with empirical corpus evidence
teaching_themes_data = [
    {
        "failure_mechanism": "short_term_optimization",
        "observation": "Executives extract short-term financial gains or cost cuts at the expense of long-term brand equity, customer trust, and reinvestment.",
        "boundary_conditions": "Applies when cost reductions target core product value or customer experience. Does not apply to eliminating operational waste or non-core overhead during a crisis.",
        "counterexample": "Crocs trimmed overcapacity and retail footprint during the Great Recession without compromising product quality, enabling a later recovery.",
        "additional_evidence_needed": "Longitudinal margin vs. R&D/reinvestment ratios, and customer retention metrics following cost-cutting announcements."
    },
    {
        "failure_mechanism": "market_or_demographic_shift",
        "observation": "Core customer demographics or consumption habits shift away, leaving legacy business models stranded in declining channels.",
        "boundary_conditions": "Applies when structural demand shifts occur faster than legacy asset depreciation. Does not apply if brand affinity remains portable across channels.",
        "counterexample": "Panda Express pivoted from declining shopping malls to standalone drive-thru locations, successfully capturing shifting dining habits.",
        "additional_evidence_needed": "Channel-level cohort retention rates, customer age distribution trends, and substitution rate metrics."
    },
    {
        "failure_mechanism": "cost_structure_mismatch",
        "observation": "High fixed operating costs (leases, labor contracts, complex menus) collide with fluctuating or price-sensitive revenues.",
        "boundary_conditions": "Critical in labor/real-estate intensive businesses. Less damaging in high-margin software or digital platforms with low marginal costs.",
        "counterexample": "Domino's streamlined store footprints and focused on lean delivery infrastructure to maintain low overhead relative to Pizza Hut.",
        "additional_evidence_needed": "Fixed vs. variable cost ratios, store-level break-even volume metrics, and margin sensitivity analysis."
    },
    {
        "failure_mechanism": "incentive_misalignment",
        "observation": "Organizational incentives (private equity hold periods, executive bonuses, contract structures) reward behaviors that undermine entity health.",
        "boundary_conditions": "Applies when principal-agent divergence exists across decision-makers and long-term stakeholders.",
        "counterexample": "Family-owned or founder-led entities (e.g. Panda Express under the Cherng family) maintaining multi-decade horizons.",
        "additional_evidence_needed": "Executive compensation structures, investor exit timeframes, and partner/franchisee profit-sharing terms."
    },
    {
        "failure_mechanism": "governance_failure",
        "observation": "Lack of independent oversight, suppressed safety/operational warnings, or board neglect leads to unchecked risk-taking or operational catastrophe.",
        "boundary_conditions": "Applies in complex, mission-critical, or regulated environments where tail risks are catastrophic.",
        "counterexample": "Regulated entities with independent audit committees and empowered safety engineering teams.",
        "additional_evidence_needed": "Whistleblower log records, internal audit independence metrics, and board composition/expertise data."
    }
]

teaching_df = pd.DataFrame(teaching_themes_data)

# Compute passage counts and subject list for each theme from corpus data
stats_per_theme = (
    mechanisms.groupby("failure_mechanism")
    .agg(
        passage_count=("subject", "count"),
        cases=("subject", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

candidate_teaching_themes = pd.merge(
    stats_per_theme,
    teaching_df,
    on="failure_mechanism",
    how="inner"
).sort_values("passage_count", ascending=False)

display(candidate_teaching_themes[[
    "failure_mechanism",
    "passage_count",
    "cases",
    "observation",
    "boundary_conditions",
    "counterexample",
    "additional_evidence_needed"
]])
